# 附录A：T.simd API参考

本附录按 API 介绍课程使用的 13 个 SIMD 接口。编写算子时，可以从索引直接找到对应接口，查看**功能、原型、参数、返回值、约束和简短示例**。

<small>适用范围：昇腾950。示例侧重 float32；涉及其他类型时会单独说明。这里的 13 个接口不是完整的 SIMD API 列表。</small>

<a id="data-path"></a>
## 先看数据在哪里，计算在哪里

**V 核是执行向量任务的整个核，UB 和向量寄存器是它内部的存储资源，Vector 计算单元负责运算。** GM 是设备侧的全局内存，本课程中对应 HBM，位于 V 核之外。

![单个V核中的SIMD数据流：GM经MTE2搬入UB，vld装入寄存器，Vector计算后由vsts和MTE3逐级写回](images/simd_dataflow.png)

<small>图A-1：蓝色箭头表示读入，橙色箭头表示计算结果逐级写回，虚线表示掩码控制。输入区和输出区共用同一个 UB；图中省略了本附录不涉及的硬件单元。</small>

以 `y = x + x` 为例，代码与图中的五步对应如下：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">步骤</th>
<th style="text-align: left; vertical-align: top;">TileLang 代码</th>
<th style="text-align: left; vertical-align: top;">硬件上发生的操作</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">① 搬入</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(X, x_ub)</code></td>
<td style="text-align: left; vertical-align: top;">MTE2 将输入从 GM 搬到当前 V 核的 UB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">② 装载</td>
<td style="text-align: left; vertical-align: top;"><code>x = S.vld(x_ub[0])</code></td>
<td style="text-align: left; vertical-align: top;">将 UB 中的一组数据装入向量寄存器</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">③ 计算</td>
<td style="text-align: left; vertical-align: top;"><code>y = S.vadd(x, x, full)</code></td>
<td style="text-align: left; vertical-align: top;">Vector 计算单元读取寄存器操作数，执行加法，把结果送回寄存器</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">④ 存储</td>
<td style="text-align: left; vertical-align: top;"><code>S.vsts(y_ub[0], y, full)</code></td>
<td style="text-align: left; vertical-align: top;">将寄存器结果存入 UB 的输出区</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">⑤ 搬出</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(y_ub, Y)</code></td>
<td style="text-align: left; vertical-align: top;">MTE3 将输出从 UB 搬回 GM</td>
</tr>
</tbody>
</table>

**连续几步 SIMD 计算可以直接使用寄存器中的中间结果。** 例如先加法、再乘法，两步之间不必先写回 UB。需要输出时，再按“寄存器 → UB → GM”的路径搬出；仅调用计算 API 不会自动更新 GM 中的输出张量。

### 单个 V 核的存储规格

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">资源</th>
<th style="text-align: left; vertical-align: top;">950 SIMD 编程中的规格</th>
<th style="text-align: left; vertical-align: top;">写代码时怎样理解</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">UB</td>
<td style="text-align: left; vertical-align: top;">每个 V 核独享，总容量 <strong>256 KB</strong>，普通对齐访问以 <strong>32 B</strong> 为边界</td>
<td style="text-align: left; vertical-align: top;"><code>x_ub</code>、<code>y_ub</code> 等缓冲区共同占用这部分空间</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">向量数据寄存器</td>
<td style="text-align: left; vertical-align: top;">数量上限 <strong>32 个</strong>，每个 <strong>256 B（2048 bit）</strong></td>
<td style="text-align: left; vertical-align: top;">保存 <code>vld</code> 读入的数据和计算结果；寄存器分配、复用由编译器处理</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">掩码寄存器</td>
<td style="text-align: left; vertical-align: top;">数量上限 <strong>8 个</strong>，每个 <strong>32 B（256 bit）</strong></td>
<td style="text-align: left; vertical-align: top;">保存 <code>pset</code> 或比较指令产生的条件，用来控制有效元素或选择数据</td>
</tr>
</tbody>
</table>

一个 256 B 向量寄存器能容纳的元素数为：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">元素类型</th>
<th style="text-align: left; vertical-align: top;">单个元素大小</th>
<th style="text-align: left; vertical-align: top;">一个寄存器中的元素数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>float32</code></td>
<td style="text-align: left; vertical-align: top;">4 B</td>
<td style="text-align: left; vertical-align: top;"><strong>64</strong></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>float16</code> / <code>bfloat16</code></td>
<td style="text-align: left; vertical-align: top;">2 B</td>
<td style="text-align: left; vertical-align: top;"><strong>128</strong></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">8 bit 类型（如 FP8）</td>
<td style="text-align: left; vertical-align: top;">1 B</td>
<td style="text-align: left; vertical-align: top;"><strong>256</strong></td>
</tr>
</tbody>
</table>

向量寄存器中一个数据元素所在的位置称为 **lane**。下文用“第 0 个元素”“前 8 个位置”等说法描述它。容量表不表示每个 API 都支持表中所有类型，也不表示一次类型转换会填满全部位置。

<small>容量说明：本页 1 KB = 1024 B。UB 的 256 KB 是总容量，系统预留和编译配置会影响程序实际可分配的大小；常规 SIMD 配置通常保留 8 KB，剩余 248 KB。寄存器数量是硬件资源上限，不是 Python 变量个数限制。规格依据：<a href="https://asc.gitcode.com/guide/programming_guide/advanced_programming/hardware_implementation/architecture_spec/npu_arch_3510.html">NPU架构版本3510</a>；数据路径依据：<a href="https://asc.gitcode.com/guide/programming_guide/programming_model/ai_core_simd_programming/c_pointer_programming/reg_vector_computation.html">Reg矢量计算编程</a>。</small>


## 示例约定

以下示例使用 `S` 作为 `T.simd` 的简写：

```python
import tilelang.ascend.language as T
from tilelang.ascend.language import simd as S
```

**每节只展示接口调用所需的几行代码。** 这些片段放在 `with T.SimdVF():` 内使用；UB 分配、GM 搬运和 Host 调用沿用正文课程的写法。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">名称</th>
<th style="text-align: left; vertical-align: top;">示例中的含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>x_ub</code>、<code>b_ub</code></td>
<td style="text-align: left; vertical-align: top;">已准备好输入数据的 UB 缓冲区；未另行说明时，均包含 64 个连续的 <code>float32</code> 元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>y_ub</code></td>
<td style="text-align: left; vertical-align: top;">UB 输出缓冲区；未另行说明时，可容纳 64 个 <code>float32</code> 元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>x</code>、<code>b</code>、<code>y</code></td>
<td style="text-align: left; vertical-align: top;">向量寄存器值，未另行说明时为 <code>float32x64</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>full</code></td>
<td style="text-align: left; vertical-align: top;"><code>S.pset(32)</code> 产生的掩码，允许处理全部 64 个 float32 元素</td>
</tr>
</tbody>
</table>

`float32x64` 表示“64 个 float32 组成的向量寄存器值”。原型中的 `DType` 表示元素类型，调用时可传入 `"float32"` 等类型字符串。

<a id="mask-mode"></a>
### 同一个 mask，在不同 API 中控制什么

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">API 类别</th>
<th style="text-align: left; vertical-align: top;">mask 为真</th>
<th style="text-align: left; vertical-align: top;">mask 为假</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">算术指令，使用默认 <code>MODE_ZEROING</code></td>
<td style="text-align: left; vertical-align: top;">计算对应元素</td>
<td style="text-align: left; vertical-align: top;">结果寄存器中的对应位置为 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>S.vsts</code></td>
<td style="text-align: left; vertical-align: top;">将对应元素写入 UB</td>
<td style="text-align: left; vertical-align: top;">不覆盖 UB 的对应位置，原内容保留</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>S.vsel</code></td>
<td style="text-align: left; vertical-align: top;">选取第一个输入 <code>src0</code></td>
<td style="text-align: left; vertical-align: top;">选取第二个输入 <code>src1</code></td>
</tr>
</tbody>
</table>

所以，**“只计算前 8 项”与“只写回前 8 项”是两个不同的动作**。掩码只影响接收它的那条指令，不能替之前的读取操作缩小访问范围。

<small>进阶提示：带 <code>mode</code> 的接口还可能使用 <code>MODE_MERGING</code>，其含义是保留目的寄存器中未被掩码选中的旧值。当前 TileLang 后端要求把这类结果赋给已初始化的可变 SIMD 寄存器；它不会自动保留某个输入向量。本附录的短示例统一使用默认 <code>MODE_ZEROING</code>。</small>

<a id="api-index"></a>
## API索引

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">类别</th>
<th style="text-align: left; vertical-align: top;">API</th>
<th style="text-align: left; vertical-align: top;">功能</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">数据读写</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vld">S.vld</a></td>
<td style="text-align: left; vertical-align: top;">从 UB 读取向量</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">数据读写</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vsts">S.vsts</a></td>
<td style="text-align: left; vertical-align: top;">将向量写回 UB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">常量生成</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vdup">S.vdup</a></td>
<td style="text-align: left; vertical-align: top;">将标量复制到向量各元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">算术</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vadd">S.vadd</a></td>
<td style="text-align: left; vertical-align: top;">两个向量逐元素相加</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">算术</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vmul">S.vmul</a></td>
<td style="text-align: left; vertical-align: top;">两个向量逐元素相乘</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">算术</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vadds">S.vadds</a></td>
<td style="text-align: left; vertical-align: top;">向量与标量相加</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">掩码</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-pset">S.pset</a></td>
<td style="text-align: left; vertical-align: top;">生成固定模式的掩码</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">序列生成</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vci">S.vci</a></td>
<td style="text-align: left; vertical-align: top;">生成连续序列</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">比较</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vcmp">S.vcmp</a></td>
<td style="text-align: left; vertical-align: top;">比较两个向量</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">比较</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vcmps">S.vcmps</a></td>
<td style="text-align: left; vertical-align: top;">比较向量与标量</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">选择</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vsel">S.vsel</a></td>
<td style="text-align: left; vertical-align: top;">根据掩码选择元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">归约</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vcadd">S.vcadd</a></td>
<td style="text-align: left; vertical-align: top;">将向量元素求和</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">类型转换</td>
<td style="text-align: left; vertical-align: top;"><a href="#api-vcvt">S.vcvt</a></td>
<td style="text-align: left; vertical-align: top;">转换向量元素类型</td>
</tr>
</tbody>
</table>


<a id="api-vld"></a>
## A.01 S.vld：从 UB 读取向量

### 功能说明

从指定的 **UB 起始地址**读取数据，返回一个向量寄存器值。读取长度和排列方式由元素类型、`dist` 共同决定。

### 函数原型

```python
def vld(addr, dist='NORM'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>addr</code>：UB 地址表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">指定从哪里开始读，例如 <code>x_ub[0]</code>。方括号中的偏移按<strong>缓冲区元素</strong>计数：对 float32，<code>x_ub[64]</code> 表示向后偏移 64 项，即 256 B。这里传的是起始位置，不是只读取那一个元素。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>dist</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"NORM"</code></td>
<td style="text-align: left; vertical-align: top;">指定如何把 UB 数据装入寄存器。默认连续读取一个寄存器的数据：float32 读 64 项，float16 读 128 项。<code>"BRC_B32"</code> 则把一个 32 bit 元素广播到寄存器各位置；它与普通连续读取的访问范围不同。</td>
</tr>
</tbody>
</table>


### 返回值

返回元素类型与源缓冲区对应的向量寄存器；float32 默认读取返回 `float32x64`。

### 使用约束

- 普通 `NORM` 读取需要准备完整的 256 B 源区域，并满足该读取方式的地址对齐要求。示例从分配好的 UB 起始位置读入。
- 接口没有 `mask` 或任意长度参数。即使后续只计算前 8 项，float32 的 `NORM` 读取仍需要 64 项有效空间。
- GM 输入必须先通过 `T.copy` 搬到 UB。不同 `dist` 具有不同布局和访问范围，不能只改字符串而不调整数据准备。

### 简短示例

```python
x = S.vld(x_ub[0])
full = S.pset(32)
S.vsts(y_ub[0], x, full)
```

第一行把 `x_ub` 的 64 个 float32 元素读入 `x`；最后一行原样写入 `y_ub`。数据路径为 **UB → 寄存器 → UB**。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vsts"></a>
## A.02 S.vsts：将向量写回 UB

### 功能说明

把向量寄存器中的数据写入 UB。`mask` 决定哪些位置被写入，`dist` 决定以什么布局写入。

### 函数原型

```python
def vsts(addr, src, mask=None, dist='NORM_B32', extent=None): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>addr</code>：UB 地址表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">目的缓冲区的起始位置，例如 <code>y_ub[0]</code>。偏移按目标缓冲区的元素数计，而非字节数。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">要存储的寄存器数据，例如 <code>vadd</code> 返回的 <code>y</code>。元素类型及寄存器中的有效位置必须与目标缓冲区、存储布局相匹配。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">为真的位置执行写入，为假的位置<strong>不覆盖原 UB 内容</strong>。省略时，按 <code>src</code> 的元素位宽生成全有效掩码；省略不等于自动识别尾部长度。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>dist</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"NORM_B32"</code></td>
<td style="text-align: left; vertical-align: top;">控制从寄存器取哪些位置、如何排到 UB。常见用法见下表；默认值适用于本附录的普通 float32 写入。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>extent</code>：整数或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">向编译器描述这次写入涉及的<strong>目标地址范围</strong>，单位为目标缓冲区元素。默认根据目标类型和 <code>dist</code> 推导。它不决定硬件实际写多少项，也不会生成尾部掩码，通常无需手动指定。</td>
</tr>
</tbody>
</table>

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">本附录使用的 <code>dist</code></th>
<th style="text-align: left; vertical-align: top;">写入行为</th>
<th style="text-align: left; vertical-align: top;">对应场景</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"NORM_B32"</code></td>
<td style="text-align: left; vertical-align: top;">按 32 bit 元素连续写入掩码选中的位置</td>
<td style="text-align: left; vertical-align: top;">普通 float32 结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"ONEPT_B32"</code></td>
<td style="text-align: left; vertical-align: top;">写出寄存器最低位置的一个 32 bit 元素</td>
<td style="text-align: left; vertical-align: top;"><code>vcadd</code> 的单个求和结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PK4_B32"</code></td>
<td style="text-align: left; vertical-align: top;">提取每个 32 bit 槽位中的低 8 bit，连续打包写出</td>
<td style="text-align: left; vertical-align: top;">本页 float32 → FP8、<code>part=0</code> 的转换结果</td>
</tr>
</tbody>
</table>


### 返回值

不返回新的向量数据；调用产生一次写入 UB 的操作。

### 使用约束

- 实际写入的所有位置都必须位于已分配的 UB 空间内。把 `extent` 改成 8，不能将全有效的 64 项存储变成 8 项存储。
- 掩码为假的位置保留 UB 旧值。如果目标此前未初始化，不能把这些位置当作 0。
- 结果写入 UB 后，还需由外层 `T.copy(y_ub, Y)` 搬到 GM。

### 简短示例

```python
x = S.vld(x_ub[0])
first8 = S.pset(32, "PAT_VL8")
S.vsts(y_ub[0], x, first8)
```

本例源 UB 仍须准备 64 项；存储只覆盖 `y_ub[0]`～`y_ub[7]`，后面的 UB 位置保持原内容。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vdup"></a>
## A.03 S.vdup：将标量复制到向量各元素

### 功能说明

在寄存器中生成常量向量。例如，把标量 2 复制到每个有效位置，得到 `[2, 2, …, 2]`。

### 函数原型

```python
def vdup(src, dtype_str: DType, mask=None, mode='MODE_ZEROING'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：标量表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">待复制的一个数，例如 <code>T.float32(2)</code>。这里需要标量，不是 UB 张量或另一个向量寄存器。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>dtype_str</code>：元素类型</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">决定结果中每个元素的类型。例如 <code>"float32"</code> 对应 64 项，<code>"float16"</code> 对应 128 项。它指定类型，不指定任意向量长度。标量应使用匹配的类型。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">控制哪些位置接收该标量。省略时按<strong>目标类型</strong>生成全有效掩码；float32 可显式传 <code>S.pset(32)</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">指定未被掩码选中位置的结果。默认将这些位置<strong>置零</strong>，不会保留原输入值。<code>MODE_MERGING</code> 保留的是目的寄存器旧值，使用条件见<a href="#mask-mode">掩码说明</a>。</td>
</tr>
</tbody>
</table>


### 返回值

返回 `dtype_str` 指定元素类型的向量寄存器。

### 使用约束

- 只在寄存器中生成数值，不分配 UB，也不自动写入 GM。
- 复制到部分位置时，默认模式使其余结果为 0；需要全为 2 时，应使用全有效掩码。

### 简短示例

```python
full = S.pset(32)
two = S.vdup(T.float32(2), "float32", full)
S.vsts(y_ub[0], two, full)
```

`two` 包含 64 个 float32 数值 2，最后写入 UB。无需提前准备一个装满 2 的 GM 张量。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vadd"></a>
## A.04 S.vadd：两个向量逐元素相加

### 功能说明

对掩码选中的每个位置，计算 `y[i] = x[i] + b[i]`。给所有元素加同一个标量时，可使用 `S.vadds`。

### 函数原型

```python
def vadd(src0, src1, mask=None, mode='MODE_ZEROING'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src0</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">第一个加数，例如从 <code>x_ub</code> 读出的 <code>x</code>。它提供每个位置的左侧操作数。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src1</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">第二个加数，例如 <code>b</code>。与 <code>src0</code> 的元素类型和长度相同，按相同下标一一配对；不进行标量广播。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">决定哪些位置参加计算。传入 <code>S.pset(32)</code> 时，64 个 float32 元素全部有效；传入 <code>S.pset(32, "PAT_VL8")</code> 时，只有前 8 项有效。省略或传 <code>None</code> 时，按输入元素位宽生成全有效掩码。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">指定未被掩码选中位置的结果。默认将这些位置<strong>置零</strong>，不会保留原输入值。<code>MODE_MERGING</code> 保留的是目的寄存器旧值，使用条件见<a href="#mask-mode">掩码说明</a>。</td>
</tr>
</tbody>
</table>


### 返回值

返回与 `src0` 类型相同的向量寄存器值；示例为 `float32x64`。

### 使用约束

- 输入必须是向量寄存器值。UB 中的数据应先由 `S.vld` 读入；不能直接把整个 UB 缓冲区当作寄存器参数传入。
- 运算不会原地改写输入。后续需使用返回的结果值，并在需要时调用 `S.vsts` 写回。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
b = S.vld(b_ub[0])
y = S.vadd(x, b, full)
S.vsts(y_ub[0], y, full)
```

若某位置的 `x` 为 3、`b` 为 2，该位置的结果就是 5。不同位置分别相加。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vmul"></a>
## A.05 S.vmul：两个向量逐元素相乘

### 功能说明

对掩码选中的每个位置，计算 `y[i] = x[i] * b[i]`。这是逐元素乘法，不执行矩阵乘，也不把乘积求和。

### 函数原型

```python
def vmul(src0, src1, mask=None, mode='MODE_ZEROING'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src0</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">第一个乘数，例如从 <code>x_ub</code> 读出的 <code>x</code>。它提供每个位置的左侧操作数。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src1</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">第二个乘数，例如 <code>b</code>。与 <code>src0</code> 的元素类型和长度相同，按相同下标一一配对；不进行标量广播。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">决定哪些位置参加计算。传入 <code>S.pset(32)</code> 时，64 个 float32 元素全部有效；传入 <code>S.pset(32, "PAT_VL8")</code> 时，只有前 8 项有效。省略或传 <code>None</code> 时，按输入元素位宽生成全有效掩码。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">指定未被掩码选中位置的结果。默认将这些位置<strong>置零</strong>，不会保留原输入值。<code>MODE_MERGING</code> 保留的是目的寄存器旧值，使用条件见<a href="#mask-mode">掩码说明</a>。</td>
</tr>
</tbody>
</table>


### 返回值

返回与 `src0` 类型相同的向量寄存器值；示例为 `float32x64`。

### 使用约束

- 输入必须是向量寄存器值。UB 中的数据应先由 `S.vld` 读入；不能直接把整个 UB 缓冲区当作寄存器参数传入。
- 运算不会原地改写输入。后续需使用返回的结果值，并在需要时调用 `S.vsts` 写回。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
b = S.vld(b_ub[0])
y = S.vmul(x, b, full)
S.vsts(y_ub[0], y, full)
```

若某位置的 `x` 为 3、`b` 为 2，该位置的结果就是 6。不同位置分别相乘。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vadds"></a>
## A.06 S.vadds：向量与标量相加

### 功能说明

让每个有效元素加上同一个标量，即 `y[i] = x[i] + scalar`。

### 函数原型

```python
def vadds(src, scalar, mask=None, mode='MODE_ZEROING'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">提供每个位置原有的数值。例如 <code>S.vld</code> 读出的 float32 向量。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>scalar</code>：标量表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">所有有效位置共用的加数。对 float32 向量，写成 <code>T.float32(1)</code> 可明确类型；不需要先用 <code>vdup</code> 生成常量向量。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">决定哪些位置参加计算。传入 <code>S.pset(32)</code> 时，64 个 float32 元素全部有效；传入 <code>S.pset(32, "PAT_VL8")</code> 时，只有前 8 项有效。省略或传 <code>None</code> 时，按输入元素位宽生成全有效掩码。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">指定未被掩码选中位置的结果。默认将这些位置<strong>置零</strong>，不会保留原输入值。<code>MODE_MERGING</code> 保留的是目的寄存器旧值，使用条件见<a href="#mask-mode">掩码说明</a>。</td>
</tr>
</tbody>
</table>


### 返回值

返回与 `src` 类型相同的向量寄存器。

### 使用约束

- 标量类型应与向量元素类型匹配。第二个参数若是另一个向量，应改用 `S.vadd`。
- 部分掩码下，默认模式把未选中位置的结果置零，不会让这些位置继续保存原 `src` 值。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
y = S.vadds(x, T.float32(1), full)
S.vsts(y_ub[0], y, full)
```

例如，输入开头的 `[-3, -2, -1, 0]` 对应结果 `[-2, -1, 0, 1]`。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-pset"></a>
## A.07 S.pset：生成固定模式的掩码

### 功能说明

在掩码寄存器中生成一组开关，指定后续指令的哪些元素位置有效。

### 函数原型

```python
def pset(elem_width: int, dist: str='PAT_ALL'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>elem_width</code>：整数，单位 bit</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;"><strong>单个数据元素的位宽</strong>。本页 float32 用 <code>32</code>，float16 / bfloat16 用 <code>16</code>，8 bit 类型用 <code>8</code>。例如 <code>32</code> 对应一个寄存器里的 64 个 float32 位置，不是“只处理 32 项”。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>dist</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"PAT_ALL"</code></td>
<td style="text-align: left; vertical-align: top;">选择开关的固定分布模式。默认让所有位置有效；<code>"PAT_VL8"</code> 只启用前 8 项。常用模式见下表，名称区分大小写。</td>
</tr>
</tbody>
</table>

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;"><code>dist</code> 模式</th>
<th style="text-align: left; vertical-align: top;">哪些元素有效</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_ALL"</code></td>
<td style="text-align: left; vertical-align: top;">所有元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_ALLF"</code></td>
<td style="text-align: left; vertical-align: top;">所有元素均无效</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_VL1"</code>、<code>"PAT_VL2"</code>、<code>"PAT_VL3"</code>、<code>"PAT_VL4"</code></td>
<td style="text-align: left; vertical-align: top;">前 1、2、3、4 项</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_VL8"</code>、<code>"PAT_VL16"</code>、<code>"PAT_VL32"</code>、<code>"PAT_VL64"</code>、<code>"PAT_VL128"</code></td>
<td style="text-align: left; vertical-align: top;">前 8、16、32、64、128 项；应与当前元素位宽对应的容量匹配</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_H"</code>、<code>"PAT_Q"</code></td>
<td style="text-align: left; vertical-align: top;">低半部分、低四分之一；float32 下分别为前 32 项、前 16 项</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"PAT_M3"</code>、<code>"PAT_M4"</code></td>
<td style="text-align: left; vertical-align: top;">从下标 0 开始，选中下标为 3 或 4 的倍数的位置</td>
</tr>
</tbody>
</table>


### 返回值

返回掩码寄存器，DSL 类型为 `boolx256`。它是指令控制条件，不是普通数值向量。

### 使用约束

- 掩码位宽应符合使用它的指令。对普通 float32 运算用 `32`；类型转换的规则另见 `vcvt`。
- `PAT_VL` 后不能随意填写长度。本页使用接口支持的固定模式；需要动态尾部长度时，应另选相应的动态掩码接口。
- `vld` 不接收掩码。创建前 8 项有效的掩码，不会让先前的整寄存器读取缩短。

### 简短示例

```python
full = S.pset(32)
first8 = S.pset(32, "PAT_VL8")
x = S.vci(T.float32(0), "float32")
y = S.vadds(x, T.float32(100), first8)
S.vsts(y_ub[0], y, full)
```

`x` 为 0～63。加法只处理前 8 项，得到 100～107；默认 `MODE_ZEROING` 让其余结果为 0。最后用 `full` 把全部 64 项写回。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vci"></a>
## A.08 S.vci：生成连续序列

### 功能说明

直接在寄存器中生成步长为 +1 或 −1 的数值序列，不需要从 GM 准备一份索引数组。

### 函数原型

```python
def vci(index, dtype_str: DType, order='INC_ORDER'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>index</code>：标量表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">序列起点，即第 0 项的值。例如 <code>T.float32(10)</code> 表示从数值 10 开始；它不是 UB 地址，也不是要生成的元素数。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>dtype_str</code>：元素类型</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">结果元素的类型，例如 <code>"float32"</code> 或 <code>"int32"</code>。类型决定可容纳的元素数，起点应采用相匹配的标量类型。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>order</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"INC_ORDER"</code></td>
<td style="text-align: left; vertical-align: top;"><code>"INC_ORDER"</code> 按 <code>index + i</code> 生成；<code>"DEC_ORDER"</code> 按 <code>index - i</code> 生成。<code>i</code> 从 0 开始，接口没有自定义步长参数。</td>
</tr>
</tbody>
</table>


### 返回值

返回指定元素类型的数值向量；float32 对应 64 项。

### 使用约束

- 没有任意长度或掩码参数；生成一整个向量后，可给后续计算、存储设置掩码。
- 它返回数值，不返回谓词。要判断“哪些值大于 0”，应再调用比较接口。

### 简短示例

```python
full = S.pset(32)
ramp = S.vci(T.float32(10), "float32", order="INC_ORDER")
S.vsts(y_ub[0], ramp, full)
```

本例得到 10～73。若仅把 `order` 改成 `"DEC_ORDER"`，序列则为 10、9、8、…、−53。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vcmp"></a>
## A.09 S.vcmp：比较两个向量

### 功能说明

比较两个寄存器的对应元素，生成掩码。例如 `op="gt"` 判断 `src0[i] > src1[i]`。

### 函数原型

```python
def vcmp(src0, src1, mask=None, op='eq'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src0</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">比较关系左侧的数值，如 <code>x &gt; b</code> 中的 <code>x</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src1</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">比较关系右侧的数值，如 <code>x &gt; b</code> 中的 <code>b</code>。元素类型和长度应与 <code>src0</code> 相同。交换两者会改变大小比较的含义。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">限定参加比较的元素位置。省略时按 <code>src0</code> 元素位宽生成全有效掩码。本例使用全有效掩码，结果供下一条选择指令使用。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>op</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"eq"</code></td>
<td style="text-align: left; vertical-align: top;">选择比较关系，见下表。默认判断相等；要选出较大值时，示例明确写 <code>op="gt"</code>。</td>
</tr>
</tbody>
</table>

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;"><code>op</code></th>
<th style="text-align: left; vertical-align: top;">比较关系</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"eq"</code> / <code>"ne"</code></td>
<td style="text-align: left; vertical-align: top;">等于 / 不等于</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"gt"</code> / <code>"ge"</code></td>
<td style="text-align: left; vertical-align: top;">大于 / 大于等于</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"lt"</code> / <code>"le"</code></td>
<td style="text-align: left; vertical-align: top;">小于 / 小于等于</td>
</tr>
</tbody>
</table>


### 返回值

返回记录比较条件的掩码寄存器 `boolx256`；不是由 float32 数值 0、1 组成的向量。

### 使用约束

- 本页示例使用有限 float32 数值；不要据此推断 NaN 等特殊值的行为。
- 比较只产生条件，不搬运数值，也不直接产生两个输入的最大值。需要选值时，继续调用 `S.vsel`。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
b = S.vld(b_ub[0])
p = S.vcmp(x, b, full, op="gt")
y = S.vsel(x, b, p)
```

若某位置 `x=3`、`b=2`，比较条件为真，`vsel` 取 3；若 `x=1`、`b=4`，条件为假，取 4。`y` 仍在寄存器中。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vcmps"></a>
## A.10 S.vcmps：比较向量与标量

### 功能说明

把向量的每个有效元素与同一个标量比较。例如 `op="gt"`、`scalar=0` 用来判断哪些元素是正数。

### 函数原型

```python
def vcmps(src, scalar, mask=None, op='lt'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">比较关系左侧的向量，如 <code>x[i] &gt; 0</code> 中的 <code>x</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>scalar</code>：标量表达式</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">右侧的公共比较值。对 float32 向量可写 <code>T.float32(0)</code>；类型应与向量元素类型匹配。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">控制哪些位置参加比较。省略时按 <code>src</code> 的元素位宽生成全有效掩码。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>op</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"lt"</code></td>
<td style="text-align: left; vertical-align: top;">可选 <code>"eq"</code>、<code>"ne"</code>、<code>"gt"</code>、<code>"ge"</code>、<code>"lt"</code>、<code>"le"</code>，分别对应等于、不等于、大于、大于等于、小于、小于等于。<strong>默认是小于</strong>，与 <code>vcmp</code> 默认的相等不同。</td>
</tr>
</tbody>
</table>


### 返回值

返回掩码寄存器 `boolx256`，可继续传给选择指令。

### 使用约束

- 第二个参数是标量；与另一个向量比较时，使用 `S.vcmp`。
- 比较条件不是可直接按 float32 写出的 0/1 数据。本节通过 `vsel` 使用这个条件。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
p = S.vcmps(x, T.float32(0), full, op="gt")
zero = S.vdup(T.float32(0), "float32", full)
y = S.vsel(x, zero, p)
```

先判断 `x > 0`，再在条件成立的位置保留 `x`，其余位置选择 0。因此 `y` 是逐元素 ReLU 的结果。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vsel"></a>
## A.11 S.vsel：根据掩码选择元素

### 功能说明

对每个位置执行 `y[i] = src0[i] if mask[i] else src1[i]`，在两个候选向量中选取数值。

### 函数原型

```python
def vsel(src0, src1, mask): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src0</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">条件为<strong>真</strong>时选取的向量。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src1</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">条件为<strong>假</strong>时选取的向量。元素类型和长度应与 <code>src0</code> 相同。想在条件为假时填零，应显式传入零向量。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填，无默认值</td>
<td style="text-align: left; vertical-align: top;">逐位置的选择条件，通常由 <code>vcmp</code>、<code>vcmps</code> 或 <code>pset</code> 生成。这里的假表示“取第二个输入”，不表示跳过该位置。</td>
</tr>
</tbody>
</table>


### 返回值

返回与 `src0` 类型相同的向量寄存器，每个位置均从两个输入之一取得数值。

### 使用约束

- 两个候选向量都应已经准备好；该接口不代替 Python 条件分支，也不进行延迟求值。
- 没有 `mode` 参数。不要把算术指令的“掩码为假时置零”规则套到这里。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
b = S.vld(b_ub[0])
p = S.vcmp(x, b, full, op="gt")
y = S.vsel(x, b, p)
S.vsts(y_ub[0], y, full)
```

比较得到 `x > b` 的位置，再真选 `x`、假选 `b`，因此写回的是逐元素最大值。若交换 `vsel` 的两个候选输入，含义也随之改变。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vcadd"></a>
## A.12 S.vcadd：将向量元素求和

### 功能说明

在本页的 float32 用法中，对一个寄存器中被掩码选中的元素求和，将和放在结果寄存器的**最低元素位置**。

### 函数原型

```python
def vcadd(src, mask=None, mode='MODE_ZEROING'): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">参加归约的一组数。本例为 <code>float32x64</code>；归约范围限于当前寄存器，不会自动遍历 GM 中的整行或整个张量。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">决定把哪些元素加进总和。省略时全部有效；传入 <code>S.pset(32, "PAT_VL8")</code> 时，只对前 8 个 float32 元素求和。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">结果写入模式。本页使用默认值，并只取最低位置的归约结果；不把返回寄存器的其他位置当作独立求和结果。</td>
</tr>
</tbody>
</table>


### 返回值

本例返回 float32 向量寄存器，最低元素位置保存一个标量和。当前接口对 `int16` / `uint16` 输入会把返回元素类型提升为 `int32` / `uint32`，不能直接套用 float32 的类型说明。

### 使用约束

- 写出一个 float32 求和结果时，使用 `dist="ONEPT_B32"`；目标 UB 至少能容纳一个 float32 元素。
- 不能用默认整向量存储，把结果写入只有一个元素的 UB 缓冲区。
- 浮点归约的计算顺序会影响舍入误差，与 Host 结果比较时应采用合理容差。

### 简短示例

```python
full = S.pset(32)
x = S.vld(x_ub[0])
total = S.vcadd(x, full)
S.vsts(y_ub[0], total, full, dist="ONEPT_B32")
```

本例 `x_ub` 存放 1～64，`y_ub` 只需接收一个 float32 结果。求和为 2080，单点存储把它写入 `y_ub[0]`。

<small><a href="#api-index">返回API索引</a></small>


<a id="api-vcvt"></a>
## A.13 S.vcvt：转换向量元素类型

### 功能说明

将寄存器中的元素转换为另一种类型。下面以 **64 个 float32 → 64 个 FP8（float8_e4m3fn）** 说明：数值转换与结果在寄存器中的排列需要一起考虑。

### 函数原型

```python
def vcvt(src, target_dtype: DType, mask=None, round="ROUND_R",
         sat=True, part=0, mode="MODE_ZEROING"): ...
```

### 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数与类型</th>
<th style="text-align: left; vertical-align: top; white-space: nowrap;">默认值</th>
<th style="text-align: left; vertical-align: top;">含义与填写方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>src</code>：向量寄存器</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">待转换的数据。本例为 <code>float32x64</code>，因此一次转换处理 64 个源元素。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>target_dtype</code>：元素类型</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;">必填</td>
<td style="text-align: left; vertical-align: top;">目标元素类型。本例传 <code>"float8_e4m3fn"</code>。只改变数值类型，不会把 64 项输入自动扩充为 256 项有效数据。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mask</code>：掩码寄存器或 <code>None</code></td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">None</code></td>
<td style="text-align: left; vertical-align: top;">省略时，按源、目标中<strong>较小的元素位宽</strong>生成全有效掩码。本例是 <code>min(32, 8)=8</code>，对应 <code>S.pset(8)</code>；不能照搬 float32 算术的 <code>S.pset(32)</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>round</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"ROUND_R"</code></td>
<td style="text-align: left; vertical-align: top;">控制目标类型无法精确表示输入时的舍入。<code>ROUND_R</code> 选最近的可表示值；距离相同时，选择尾数最低位为 0 的值，即“就近取偶”。<code>ROUND_Z</code> 表示向 0 舍入。可用模式受转换类型组合限制，本例保持 <code>ROUND_R</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>sat</code>：布尔值</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">True</code></td>
<td style="text-align: left; vertical-align: top;">请求底层转换的饱和模式：<code>True</code> 对应开启，<code>False</code> 对应关闭。<strong>不能把它视为统一的数值截断函数</strong>；实际溢出行为取决于转换路径。本例只处理 FP8 有限范围内的值。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>part</code>：整数</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">0</code></td>
<td style="text-align: left; vertical-align: top;">选择宽窄类型转换使用的子位置。本例 float32 → FP8 允许 0～3，分别对应每个 32 bit 槽位里的 4 个 8 bit 子位置；<code>0</code> 对应低 8 bit，便于随后用 <code>PK4_B32</code> 打包写回。它不是元素总数，也不是第几个 64 项输入块。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>mode</code>：字符串</td>
<td style="text-align: left; vertical-align: top; white-space: nowrap;"><code style="; white-space: nowrap; overflow-wrap: normal; word-break: normal;">"MODE_ZEROING"</code></td>
<td style="text-align: left; vertical-align: top;">控制未被掩码选中位置的结果。本例使用默认置零模式；使用合并模式还需满足<a href="#mask-mode">可变目的寄存器的要求</a>，且该转换路径必须支持。</td>
</tr>
</tbody>
</table>


### 返回值

本例返回 DSL 类型 `float8_e4m3fnx256`。它描述一个 256 B 寄存器的表示类型；本次只产生 64 个有效转换结果，按 `part=0` 分布在对应子位置上。

### 使用约束

- 本例输入均为有限 float32 数值，范围为 `[-448, 448]`。课程当前环境中，±500 即使使用 `sat=True` 也会转成 NaN，因此这里不依赖该开关处理越界值。
- 目标 `y_ub` 为 64 个 `float8_e4m3fn` 元素。转换后用 `PK4_B32` 取出 64 个结果，不能沿用普通 float32 的连续存储模式。
- 更换源类型、目标类型后，掩码位宽、`part` 含义、可用舍入模式和存储方式可能同时变化。例如二分转换使用 `part=0/1`，不能沿用本例的四分取值。

### 简短示例

```python
x = S.vld(x_ub[0])
q = S.vcvt(x, "float8_e4m3fn", round="ROUND_R", sat=True, part=0)
S.vsts(y_ub[0], q, dist="PK4_B32")
```

例如，1.0625 位于两个可表示值中间，按本例规则舍入为 1；1.1875 舍入为 1.25。`vcvt` 完成数值转换，`vsts` 再把分散在寄存器中的 64 个 FP8 结果打包成连续的 UB 数据。

<small><a href="#api-index">返回API索引</a></small>
